In [1]:
#NOTE: SUPPRESSES WARNINGS!

import warnings

warnings.simplefilter(action="ignore", category=RuntimeWarning)
#warnings.simplefilter(action="default", category=RuntimeWarning)

#import relevant libraries
import sys
import os
import glob

import numpy as np
import scipy as sp
import pandas as pd
import matplotlib as mpl
import seaborn as sns
import datetime as dt
import shutil
import osar

osar.__version__

'0.23.9'

In [2]:
openPath = "D:\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\DATA\\OSAR\\unproc\\"
#openPath = "C:\\Users\\lnico\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee\\DATA\\OSAR\\Unprocessed\\"
files = os.listdir(openPath)

print (files)


['MB210B x ACR', 'SS01194 x ACR', 'SS75199 x ACR', 'SS86947 x ACR', 'SS97567 x ACR', 'SS97567 x Chrimson2', 'w1118 x ACR', 'w1118 x Chrimson2', 'w1118 x MB077B [G]', 'w1118 x MB210B [G]', 'w1118 x SS01194 [G]', 'w1118 x SS75199 [G]', 'w1118 x SS86947', 'w1118 x SS86947 [G]']


In [3]:
files = ['w1118 x SS86947']

In [4]:
#renames files

for f in files:
    
    Path1 = openPath + f  #returns me E:\20210905\R58E02 x ACR --> also the file i want to save the new labels in           
    ####################################
    files2 = os.listdir(Path1)  #returns me EighthGreen, HalfGreen etc etc

    for i in files2:    #EighthGreen
        Path2 = Path1 + "\\" + i #returns me D:\Nicole Lee - HDD\PHD\Firstsetofdataruns\ACR x MB082C\EighthGreen
        #print(files2)
        files3 = os.listdir(Path2)  #returns me countlog, execution, metadata etc
        #print (files3)
        
        for j in files3: #opens each excel file
            if j.startswith("CountLog"):
           # if j.startswith("CountLog"):    
                jj= os.path.splitext(j)[0]
                renamed = jj + '_' + f.split(' [')[0] + '_' + i + ".csv"  #new file name, leaves out any [G] or [R]
                #print(renamed)
                os.chdir(Path2)
                cwd = os.getcwd()
                #print (cwd)                
                countlogfile = os.rename(j,renamed)
                
print ("Finished renaming files!")


Finished renaming files!


In [5]:
#copying/moving files

for f in files:
    
    Path1 = openPath + f  #returns me E:\20210905\R58E02 x ACR --> also the file i want to save the new labels in      
    destination = openPath + f +"\\countlog"
    isExist = os.path.exists(destination)
    if not isExist:
        os.makedirs(destination)
    #print (destination)
    ####################################
    
    files2 = os.listdir(Path1)  #returns me EighthGreen, HalfGreen etc etc

    for i in files2:    #EighthGreen
        Path2 = Path1 + "\\" + i #returns me D:\Nicole Lee - HDD\PHD\Firstsetofdataruns\ACR x MB082C\EighthGreen
        files3 = os.listdir(Path2)  #returns me countlog, execution, metadata etc
        if i != "countlog":
            for j in files3: #opens each excel file            
                if j.startswith("CountLog"):
                    jj= os.path.splitext(j)[0]
                    newfile = jj + ".csv"
                    #moving the file
                    tempfile = Path1 + "\\" + i+ "\\" + newfile
                    shutil.move(tempfile,destination)

print ("Finished moving files!")

Finished moving files!


In [ ]:
#copies renamed CountLogs from new raw date folders into DATA\OSAR (raw folders are left as they are)
import re
import filecmp

specifiedpath = "C:\\Users\\User\\NUS Dropbox\\acclab\\Nicole M Lee\\"
rawPath = specifiedpath + "Raw data files\\OSAR\\"
dataPath = specifiedpath + "DATA\\OSAR\\"

last_processed = "20260824"  #from Organizosar.md
dry_run = True               #set to False to actually copy
min_rows = 3000              #complete logs have about 3900 rows, shorter ones are aborted runs
name_fixes = {"MB57B": "MB057B", "SS76074": "SS76094", "MB399C": "MB399B"}  #typos in raw folder names

responders = {"Green": "ACR", "Red": "Chrimson2"}
wtfolders = {"Green": "WT\\WT_Green\\", "Red": "WT\\WT_Red\\"}

def fixname(folder):
    for wrong, right in name_fixes.items():
        folder = re.sub(r"\b" + wrong + r"\b", right, folder)
    return folder

def rowcount(path):
    with open(path, errors="replace") as fh:
        return sum(1 for line in fh if line.strip()) - 1

def destinations(folder, colour):
    #folder keeps any [G] or [R], genotype does not
    genotype = re.sub(r"\s*\[.\]$", "", folder)
    driver, partner = genotype.split(" x ", 1)
    if driver != "w1118":
        return [dataPath + genotype + "\\countlog\\"]
    if partner in responders.values():
        return None  #w1118 x ACR and w1118 x Chrimson2: ask where these go
    return [dataPath + partner + " x " + responders[colour] + "\\countlog\\", dataPath + wtfolders[colour] + folder + "\\"]

copied = {}
counts = {"copied": 0, "already there": 0, "aborted": 0}
aborted = []
new_crosses = set()  #cross folders that do not exist yet, most likely a typo
not_copied = []
planned = set()  #the same log can sit in two date folders

for date in sorted(d for d in os.listdir(rawPath) if re.fullmatch(r"\d{8}", d) and d > last_processed):
    for rawfolder in sorted(os.listdir(rawPath + date)):
        folder = fixname(rawfolder)
        genotype = re.sub(r"\s*\[.\]$", "", folder)
        for i in sorted(os.listdir(rawPath + date + "\\" + rawfolder)):  #Eighth_Green etc
            Path2 = rawPath + date + "\\" + rawfolder + "\\" + i
            colour = i.split("_")[-1]
            for j in sorted(os.listdir(Path2)):
                if not j.startswith("CountLog"):
                    continue
                source = Path2 + "\\" + j
                targets = destinations(folder, colour)
                if targets is None:
                    not_copied.append(("no destination", source))
                    continue
                if rowcount(source) < min_rows:
                    counts["aborted"] += 1
                    aborted.append(source)
                    continue
                if not os.path.isdir(os.path.dirname(os.path.dirname(targets[0]))):
                    new_crosses.add(os.path.basename(os.path.dirname(os.path.dirname(targets[0]))) + " (from " + date + "\\" + rawfolder + ")")
                renamed = os.path.splitext(j)[0] + "_" + genotype + "_" + i + ".csv"
                for target in targets:
                    if target + renamed in planned:
                        counts["already there"] += 1
                        continue
                    if os.path.exists(target + renamed):
                        if filecmp.cmp(source, target + renamed, shallow=False):
                            counts["already there"] += 1
                        else:
                            not_copied.append(("different file already there", target + renamed))
                        continue
                    planned.add(target + renamed)
                    counts["copied"] += 1
                    copied.setdefault(target.replace(dataPath, ""), []).append(renamed)
                    if not dry_run:
                        os.makedirs(target, exist_ok=True)
                        shutil.copy2(source, target + renamed)

print("dry run" if dry_run else "copied", counts)
for target in sorted(copied):
    print(target, len(copied[target]))
print("\nNEW crosses, check for typos:")
for c in sorted(new_crosses):
    print(c)
print("\nNot copied, needs a decision:")
for p in not_copied:
    print(p)
print("\nAborted runs, left out automatically:")
for p in aborted:
    print(p)
